DuckDB load

In [2]:
import os
import pandas as pd
import duckdb

base_dir = "/kaggle/input/datasets/shafwan26/bdc-trashclassification/SATRIA DATA/train"
kategori = ["0_Recyclable", "1_Electronic", "2_Organic"]

dataset_info = []

for label in kategori:
    folder_path = os.path.join(base_dir, label)
    # Pastikan folder ada sebelum membaca isinya
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                dataset_info.append({
                    "filepath": os.path.join(folder_path, filename),
                    "label": label
                })

# Ubah list dictionary menjadi Pandas DataFrame
df_images = pd.DataFrame(dataset_info)
print(df_images)

# Membuat file database lokal bernama 'dataset_ku.duckdb'
# (Gunakan database=':memory:' jika hanya ingin berjalan di RAM sementara)
con = duckdb.connect(database='dataset_ku.duckdb')

# Membuat tabel 'image_metadata' di DuckDB dari DataFrame Pandas
con.execute("CREATE TABLE IF NOT EXISTS image_metadata AS SELECT * FROM df_images")

print("Data berhasil dimasukkan ke DuckDB!")

query_distribusi = """
SELECT label, COUNT(*) as jumlah_gambar 
FROM image_metadata 
GROUP BY label
ORDER BY label;
"""
print(con.execute(query_distribusi).df())

Empty DataFrame
Columns: []
Index: []


InvalidInputException: Invalid Input Error: Need a DataFrame with at least one column

CNN Common

In [2]:
"""
Shared helpers untuk CNN progression (dimodifikasi untuk Kaggle & struktur folder).
"""
from __future__ import annotations

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress TF info logs
os.environ.setdefault("PYTHONHASHSEED", "0")  # freeze hash for reproducibility
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # don't allocate all GPU memory at once

import gc
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import classification_report


# === MEMORY MONITORING ===
def get_memory_usage():
    """Return current memory usage in MB (works on Linux/Kaggle)."""
    try:
        import psutil
        process = psutil.Process(os.getpid())
        mem = process.memory_info().rss / (1024 * 1024)
        return mem
    except ImportError:
        return 0.0

def print_memory_usage(label: str = ""):
    """Print current memory usage with label."""
    mem = get_memory_usage()
    if mem > 0:
        print(f"  [MEMORY] {label}: {mem:.1f} MB")

def cleanup_memory():
    """Force garbage collection to free unused memory."""
    gc.collect()
    try:
        tf.keras.backend.clear_session()
    except Exception:
        pass

# === PATH LAYOUT (Kaggle dataset format) ===
# SESUAIKAN 'nama-dataset-kamu' DENGAN NAMA DATASET ASLI DI KAGGLE
KAGGLE_DATASET_NAME = "BDC_TrashClassification"  # GANTI INI!
BASE = Path("/kaggle/working")
DATASET = Path(f"/kaggle/input/{KAGGLE_DATASET_NAME}/SATRIA DATA")

# Jika folder test ada, sesuaikan path-nya. Jika tidak ada, biarkan saja.
TRAIN_DIR = DATASET / "train" 
TEST_DIR = DATASET / "test" # Hapus atau komen jika tidak ada folder test

IMG_SIZE = 128
SEED = 42
tf.keras.utils.set_random_seed(SEED)


# === DATA LOADING ===

def _load_image(path: Path, img_size: int) -> np.ndarray:
    """Load 1 gambar, resize, normalize ke [0,1]. Return shape (img_size, img_size, 3)."""
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"gambar tidak bisa dibaca: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    return img / 255.0


def load_data_from_folders(data_dir: Path, img_size: int, has_labels: bool = True):
    """
    Load gambar + label dari struktur folder.
    Path gambar = data_dir / label / filename.
    Returns (X, y_or_None, ids, classes, class_to_idx).
    """
    print_memory_usage(f"Before loading data from {data_dir}")
    
    images = []
    labels = []
    ids = []
    classes = []
    class_to_idx = {}

    if has_labels:
        # Dapatkan daftar folder (kelas) dan buat mapping
        classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
        class_to_idx = {c: i for i, c in enumerate(classes)}
        
        count = 0
        for label_name in classes:
            label_dir = data_dir / label_name
            for img_path in label_dir.glob("*.*"): # Mengambil semua file (jpg, png, dll)
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    try:
                        img = _load_image(img_path, img_size)
                        images.append(img)
                        labels.append(class_to_idx[label_name])
                        ids.append(img_path.name)
                        count += 1
                        if count % 100 == 0:
                            print_memory_usage(f"Loaded {count} images")
                    except Exception as e:
                        print(f"Warning: Gagal memuat {img_path}: {e}")
    else:
        # Jika tidak ada label (misal data test dalam 1 folder tanpa subfolder)
        count = 0
        for img_path in data_dir.glob("*.*"):
             if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                try:
                    img = _load_image(img_path, img_size)
                    images.append(img)
                    ids.append(img_path.name)
                    count += 1
                    if count % 100 == 0:
                        print_memory_usage(f"Loaded {count} images")
                except Exception as e:
                    print(f"Warning: Gagal memuat {img_path}: {e}")

    # Konversi list ke numpy array
    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32) if has_labels else None

    print_memory_usage(f"After loading data from {data_dir}")
    cleanup_memory()
    return X, y, ids, classes, class_to_idx


def load_train_data(img_size: int = IMG_SIZE):
    """Load train data. Returns (X, y, classes, class_to_idx)."""
    X, y, _, classes, class_to_idx = load_data_from_folders(TRAIN_DIR, img_size, has_labels=True)
    return X, y, classes, class_to_idx


def load_test_public(img_size: int = IMG_SIZE):
    """Load test data. Returns (X, ids)."""
    # Pastikan folder TEST_DIR ada. Jika test data juga dalam subfolder,
    # panggil dengan has_labels=True dan modifikasi kembaliannya.
    # Asumsi test data ada di dalam 1 folder tanpa subfolder label:
    if TEST_DIR.exists():
        X, _, ids, _, _ = load_data_from_folders(TEST_DIR, img_size, has_labels=False)
        return X, ids
    else:
        print("Warning: Folder test tidak ditemukan.")
        return None, None


# === SUBMISSION & GRADING ===
# (Fungsi grading dipertahankan, tapi akan gagal jika solution.csv tidak ada di Kaggle. 
# Kamu bisa menghapusnya jika tidak dibutuhkan lagi.)

def save_submission(ids: list[str], pred_labels: list[str], out_path: Path | None = None):
    if out_path is None:
        out_path = BASE / "submission.csv"
    out_df = pd.DataFrame({
        "id": ids,
        "label": pred_labels,
    })
    out_df.to_csv(out_path, index=False)
    print(f"  [SAVED] {out_path} ({len(pred_labels)} predictions)")
    return out_path


# === UTILITAS ===

def class_weights_from_labels(y: np.ndarray, num_classes: int) -> dict:
    from sklearn.utils.class_weight import compute_class_weight
    weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=y)
    return {i: float(w) for i, w in enumerate(weights)}

CNN 07 vit

In [3]:
from __future__ import annotations
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import duckdb
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from PIL import Image
from tqdm import tqdm

# === CONFIG ===

# [!] UBAH PATH INI SESUAI DENGAN DIREKTORI DATASETMU DI KAGGLE [!]
TEST_DIR = Path("/kaggle/input/datasets/shafwan26/bdc-trashclassification/SATRIA DATA/test") 
SAMPLE_SUBMISSION_PATH = Path("/kaggle/input/datasets/shafwan26/bdc-trashclassification/submission.csv")
DUCKDB_PATH = "/kaggle/input/models/shafwan26/vit-7-without-k-fold/pytorch/default/1/dataset_ku.duckdb"

IMG_SIZE = 224  
BATCH_SIZE = 32
NUM_EPOCHS = 15 # Dibuat 15 agar tidak terlalu lama karena sekarang dilatih 5 kali (5 fold)
LR = 0.001
SEED = 42
K_FOLDS = 5 # Menggunakan 5-Fold Cross Validation
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED)
np.random.seed(SEED)

# === DATASET CLASSES ===
class TrainValDataset(Dataset):
    def __init__(self, dataframe, label_mapping, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_mapping = label_mapping

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'filepath']
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Mengembalikan gambar hitam kosong jika file rusak/hilang
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        if self.transform:
            image = self.transform(image)
            
        label_text = self.dataframe.loc[idx, 'label']
        label = self.label_mapping[label_text]
        return image, label

class TestDataset(Dataset):
    def __init__(self, dataframe, test_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.test_dir = Path(test_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_id = self.dataframe.loc[idx, 'id']
        # Pastikan ekstensi .jpg ada
        img_name = f"{img_id}.jpg" if not str(img_id).endswith('.jpg') else str(img_id)
        img_path = self.test_dir / img_name
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        if self.transform:
            image = self.transform(image)
            
        return image, img_id

# === TRANSFORMS ===
def get_transforms(train: bool = True):
    if train:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

# === MODEL ===
def build_model(num_classes: int):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_SWAG_LINEAR_V1)
    
    for param in model.conv_proj.parameters():
        param.requires_grad = False
    for param in model.encoder.parameters():
        param.requires_grad = False

    in_features = model.heads.head.in_features
    model.heads = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features=in_features, out_features=num_classes, bias=True),
    )
    return model.to(DEVICE)

# === TRAINING LOOP (PER EPOCH) ===
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return running_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return running_loss / len(loader), correct / total

# === MAIN PROCESS ===
def main():
    print("=" * 64)
    print("LEVEL 07: ViT-B/16 SWAG | 5-Fold Cross Validation & Ensemble")
    print("=" * 64)

    # 1. LOAD DATA DARI DUCKDB
    print("\n[1/4] Load data from DuckDB...")
    try:
        con = duckdb.connect(DUCKDB_PATH)
        df = con.execute("SELECT filepath, label FROM image_metadata").df()
    except Exception as e:
        print(f"Gagal memuat DuckDB: {e}")
        return

    # Sesuai screenshot sebelumnya, kelasnya: ['0_Recyclable', '1_Electronic', '2_Organic']
    classes = sorted(df['label'].unique().tolist())
    num_classes = len(classes)
    label_mapping = {c: i for i, c in enumerate(classes)}
    print(f"  Classes Mapping: {label_mapping}") # Akan otomatis jadi 0, 1, 2

    # Siapkan direktori penyimpanan model Fold
    models_dir = BASE / "models"
    models_dir.mkdir(exist_ok=True)
    
    # 2. K-FOLD CROSS VALIDATION
    print("\n[2/4] Memulai K-Fold Cross Validation Training...")
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    
    fold_model_paths = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
        print(f"\n---> FOLD {fold + 1}/{K_FOLDS} <---")
        
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        train_dataset = TrainValDataset(train_df, label_mapping, transform=get_transforms(train=True))
        val_dataset = TrainValDataset(val_df, label_mapping, transform=get_transforms(train=False))
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        
        model = build_model(num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        
        best_val_acc = 0.0
        model_save_path = models_dir / f"vit_fold_{fold + 1}.pth"
        
        for epoch in range(NUM_EPOCHS):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
            val_loss, val_acc = evaluate(model, val_loader, criterion)
            
            # Simpan model terbaik untuk fold ini
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), model_save_path)
                
            print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f})")
            
        fold_model_paths.append(model_save_path)
        print(f"Model Fold {fold+1} disimpan!")

    # 3. ENSEMBLE INFERENCE (PREDIKSI TEST DATA)
    print("\n[3/4] Memulai Soft Voting Ensemble Prediction...")
    
    test_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    test_dataset = TestDataset(test_df, TEST_DIR, transform=get_transforms(train=False))
    # Batch size prediksi bisa dibesarkan agar lebih cepat
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)

    # Load semua 5 model yang sudah di training
    ensemble_models = []
    for path in fold_model_paths:
        m = build_model(num_classes)
        m.load_state_dict(torch.load(path, map_location=DEVICE))
        m.eval()
        ensemble_models.append(m)

    predictions = []
    ids = []

    with torch.no_grad():
        for images, batch_ids in tqdm(test_loader, desc="Predicting Test Data"):
            images = images.to(DEVICE)
            
            # Matriks kosong untuk menampung total probabilitas
            batch_probs = torch.zeros((images.size(0), num_classes)).to(DEVICE)
            
            # Setiap model melakukan prediksi lalu probabilitasnya dijumlahkan
            for m in ensemble_models:
                outputs = m(images)
                probs = F.softmax(outputs, dim=1) 
                batch_probs += probs
                
            # Rata-rata probabilitas dari kelima model
            batch_probs /= K_FOLDS
            
            # Ambil index probabilitas tertinggi (akan bernilai 0, 1, atau 2)
            _, preds = torch.max(batch_probs, 1)
            
            predictions.extend(preds.cpu().numpy())
            ids.extend(batch_ids)

    # 4. SIMPAN SUBMISSION
    print("\n[4/4] Menyimpan ke submission.csv...")
    submission_df = pd.DataFrame({
        'id': ids,
        'predicted': predictions
    })
    
    submission_file = BASE / "submission.csv"
    submission_df.to_csv(submission_file, index=False)
    
    print("\n" + "=" * 64)
    print(f"SELESAI! File siap di-submit: {submission_file}")
    print("=" * 64)

if __name__ == "__main__":
    main()

LEVEL 07: ViT-B/16 SWAG | 5-Fold Cross Validation & Ensemble

[1/4] Load data from DuckDB...
Gagal memuat DuckDB: IO Error: Cannot open file "/kaggle/input/models/shafwan26/vit-7-without-k-fold/pytorch/default/1/dataset_ku.duckdb": Read-only file system
